<p style="text-align:center">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
</p>

# **SpaceX Falcon 9 First Stage Landing Prediction**

# Lab 1: Collecting the Data

Estimated time needed: **45** minutes

In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch.

## Objectives
- Request to the SpaceX API
- Clean the requested data


## Import Libraries and Define Auxiliary Functions

In [ ]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all columns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully!")

Libraries imported successfully!


Below we define a series of helper functions that use the API to extract information using identification numbers in the launch data.

In [ ]:
# Global variables
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

In [ ]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x))
            if response.status_code == 200 and response.text.strip():
                BoosterVersion.append(response.json()['name'])
            else:
                BoosterVersion.append(None)

In [ ]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

In [ ]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/" + load).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

In [ ]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API:

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print("Response status:", response.status_code)

Response status: 200


### Task 1: Request and parse the SpaceX launch data using the GET request

To make the requested JSON results more consistent, we will use the following static response object for this project:

In [ ]:
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url)
response.status_code

200

In [ ]:
# Use json_normalize method to convert the json result into a dataframe
data = pd.json_normalize(response.json())

# Check the shape: (rows, columns)
print("Shape:", data.shape)

Shape: (90, 43)


In [ ]:
# Get the head of the dataframe
data.head(5)

   flight_number                date_utc                    rocket  \
0              1  2006-03-24T22:30:00.000Z  5e9d0d95eda69955f709d1eb   
1              2  2007-03-21T01:10:00.000Z  5e9d0d95eda69955f709d1eb   
2              3  2008-08-03T03:34:00.000Z  5e9d0d95eda69955f709d1eb   
3              4  2008-09-28T23:15:00.000Z  5e9d0d95eda69955f709d1eb   
4              5  2009-07-13T03:35:00.000Z  5e9d0d95eda69955f709d1eb   

(... 43 columns total — IDs for rocket, payloads, launchpad, cores, etc.)

You will notice that a lot of the data are IDs. We will now use the API to get information about the launches using those IDs.

In [ ]:
# Take a subset keeping only the features we want and flight number, and date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy boosters) and multiple payloads
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Extract the single value in the list
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert date_utc to datetime and extract the date
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print("Filtered data shape:", data.shape)
data.head()

Filtered data shape: (90, 7)


In [ ]:
# Before calling API helper functions, show BoosterVersion is empty
BoosterVersion

[]

In [ ]:
# Call getBoosterVersion
getBoosterVersion(data)
print("BoosterVersion populated with", len(BoosterVersion), "entries")

BoosterVersion populated with 90 entries


In [ ]:
BoosterVersion[0:5]

['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1']

In [ ]:
# Call getLaunchSite
getLaunchSite(data)
print("LaunchSite entries:", len(LaunchSite))

LaunchSite entries: 90


In [ ]:
# Call getPayloadData
getPayloadData(data)
print("PayloadMass entries:", len(PayloadMass))

PayloadMass entries: 90


In [ ]:
# Call getCoreData
getCoreData(data)
print("Outcome entries:", len(Outcome))

Outcome entries: 90


Finally, let's construct our dataset using the data we have obtained:

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

In [ ]:
# Create a dataframe from launch_dict
df = pd.DataFrame(launch_dict)
print("DataFrame shape:", df.shape)

DataFrame shape: (90, 17)


In [ ]:
# Show the head of the dataframe
df.head()

   FlightNumber        Date BoosterVersion  PayloadMass Orbit  \
0             1  2006-03-24       Falcon 1         20.0   LEO   
1             2  2007-03-21       Falcon 1          NaN   LEO   
2             3  2008-08-03       Falcon 1          NaN   LEO   
3             4  2008-09-28       Falcon 1        165.0   LEO   
4             5  2009-07-13       Falcon 1          NaN   LEO   

        LaunchSite    Outcome  Flights  GridFins  Reused   Legs LandingPad  \
0  Kwajalein Atoll  None None        1     False   False  False       None   
1  Kwajalein Atoll  None None        1     False   False  False       None   
2  Kwajalein Atoll  None None        1     False   False  False       None   
3  Kwajalein Atoll  None None        1     False   False  False       None   
4  Kwajalein Atoll  None None        1     False   False  False       None   

   Block  ReusedCount   Serial  Longitude  Latitude  
0   None            0  Merlin1A   -154.808     9.047  
1   None            0  Merlin2A

### Task 2: Filter the dataframe to only include `Falcon 9` launches

Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches.
Filter the data dataframe using the `BoosterVersion` column to only keep the Falcon 9 launches.
Save the filtered data to a new dataframe called `data_falcon9`.

In [ ]:
# Filter to Falcon 9 only
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].reset_index(drop=True)
print("Falcon 9 launches:", len(data_falcon9))
data_falcon9.head()

Falcon 9 launches: 85


In [ ]:
# Reset the FlightNumber column sequentially
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9.head(10)

   FlightNumber        Date BoosterVersion  PayloadMass Orbit LaunchSite  \
0             1  2010-06-04       Falcon 9          NaN   LEO  CCAFS SLC 40   
1             2  2010-12-08       Falcon 9          NaN   LEO  CCAFS SLC 40   
2             3  2012-05-22       Falcon 9        525.0   LEO  CCAFS SLC 40   
3             4  2012-10-08       Falcon 9        500.0   LEO  CCAFS SLC 40   
4             5  2013-03-01       Falcon 9        677.0   ISS  CCAFS SLC 40   
...

## Data Wrangling

We can see below that some of the rows are missing values in our dataset.

In [ ]:
data_falcon9.isnull().sum()

FlightNumber      0
Date              0
BoosterVersion    0
PayloadMass       5
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad       26
Block             3
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64

### Task 3: Dealing with Missing Values

Calculate below the mean for the `PayloadMass` using the `.mean()`. Then use the mean and the `.replace()` function to replace `np.nan` values in the data with the mean you calculated.

The `LandingPad` column will retain None values to represent when landing pads were not used.

In [ ]:
# Calculate the mean value of PayloadMass column
mean_payload_mass = data_falcon9['PayloadMass'].mean()
print(f"Mean PayloadMass: {mean_payload_mass:.2f} kg")

# Replace the np.nan values with the mean value
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, mean_payload_mass)

# Verify no more NaN in PayloadMass
print("\nMissing values after imputation:")
print(data_falcon9.isnull().sum())

Mean PayloadMass: 4630.68 kg

Missing values after imputation:
FlightNumber      0
Date              0
BoosterVersion    0
PayloadMass       0
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad       26
Block             3
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64


You should see the number of missing values of the `PayLoadMass` change to zero.

Now we should have no missing values in our dataset except for in `LandingPad`.

We can now export it to a **CSV** for the next section:

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Dataset exported to 'dataset_part_1.csv'")
print("Final dataset shape:", data_falcon9.shape)
data_falcon9.dtypes

Dataset exported to 'dataset_part_1.csv'
Final dataset shape: (85, 17)


## Summary

In this lab, we:
1. **Requested** SpaceX launch data from a REST API and parsed the JSON response into a Pandas DataFrame using `pd.json_normalize()`
2. **Cleaned** the raw data by:
   - Keeping only relevant columns (`rocket`, `payloads`, `launchpad`, `cores`, `flight_number`, `date_utc`)
   - Removing multi-core and multi-payload launches
   - Filtering to launches on or before November 13, 2020
3. **Enriched** the data by calling helper functions (`getBoosterVersion`, `getLaunchSite`, `getPayloadData`, `getCoreData`) that made additional API calls to fill in human-readable details
4. **Filtered** to Falcon 9 launches only (removing 5 Falcon 1 records) → **85 Falcon 9 launches**
5. **Imputed** missing `PayloadMass` values with the column mean (~4,630 kg); `LandingPad` retains `None` where no pad was used
6. **Exported** the cleaned dataset to `dataset_part_1.csv` for use in subsequent labs

### Key Observations
- The dataset covers Falcon 9 launches from **2010 to November 2020**
- There are **85 Falcon 9 launches** after filtering
- Landing attempts began around **2013–2014** with early ocean landings
- Successful ASDS and RTLS landings became more common from **2016 onward**
- Block 5 boosters (introduced ~2018) show the highest reuse counts
